# Run this project on a Colab GPU

Use this notebook to set up the repository and train with Colab's GPU. In Colab, first select **Runtime > Change runtime type > GPU**. The project currently supports Python 3.11 and 3.12.

In [ ]:
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(f"This setup expects Python 3.11 or 3.12; Colab has {sys.version.split()[0]}.")

print(f"Python {sys.version.split()[0]} is within the supported project range.")

## Get the repository

Clone the public GitHub repository into Colab. The cell reuses the checkout if you run it again.

In [ ]:
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/llm-internals")
REPO_URL = "https://github.com/KrishnaTarun/llm-internals.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

%cd /content/llm-internals

## Install GPU PyTorch

`pyproject.toml` pins `torch==2.2.2` and `torchvision==0.17.2`, and configures **uv** to use PyTorch's CPU-only index. Do not run `uv sync` in this Colab setup. Install the matching CUDA 12.1 wheels explicitly with pip instead; pip does not use the `tool.uv.sources` setting.

In [ ]:
%pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.2.2 torchvision==0.17.2

## Install the remaining project dependencies

This reads `[project].dependencies` from `pyproject.toml` and installs those requirements, excluding PyTorch packages already installed from the CUDA index. It therefore keeps the other package constraints in the TOML without replacing Colab's CUDA PyTorch with the CPU build.

In [ ]:
%pip install packaging

import subprocess
import sys
import tomllib
from packaging.requirements import Requirement

with open("pyproject.toml", "rb") as project_file:
    project_dependencies = tomllib.load(project_file)["project"]["dependencies"]

pytorch_packages = {"torch", "torchvision"}
other_dependencies = [
    dependency
    for dependency in project_dependencies
    if Requirement(dependency).name.lower() not in pytorch_packages
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", *other_dependencies],
    check=True,
)

## Verify CUDA

Run this before training. If it reports that CUDA is unavailable, check that the Colab runtime is set to GPU, then restart the runtime and rerun the setup cells.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime in Colab and restart it.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

## Train GPT-style

The supplied config points to `output/checkpoints/GPTstyle/model.pt`; when that checkpoint exists, training resumes from it. Set `model.resume_from: null` in `configs/gptstyle_training.yaml` to start fresh. Colab storage is temporary, so download or copy the checkpoint to Drive if you need to keep it after the session.

In [ ]:
!python src/train.py --config configs/gptstyle_training.yaml